## Ligand Contribution Check via Reverse Score-Binder

This notebook estimates how much a ligand contributes to MaSIF match quality for a given `(target, matched_protein)` pair.

### Core idea
For a hit from MaSIF search, compare reverse-mode scores in two conditions:
1. **With ligand**: score `matched_protein` against the original `target`.
2. **Without ligand**: score the same `matched_protein` against `target-no-ligand`.

A large positive delta (`with_ligand - no_ligand`) suggests ligand-dependent matching.

### Why reverse mode?
The CSV row stores a transform that places the matched protein in the target frame. To reverse the query direction, we invert that transform so we can score from `matched_protein` back onto the target side consistently.

### What this notebook does
1. Load one row from the search results CSV.
2. Reproduce row-like scores using `masif_search.py --score_binder --transform`.
3. Invert the stored transform and run reverse scoring.
4. Preprocess `target-no-ligand`.
5. Run reverse scoring again against `target-no-ligand`.
6. Compute:
   - `delta_nn_score`
   - `delta_desc_dist_score`
   - `delta_mean_desc_dist_score`

### Assumptions
- `masif_search.py` supports `--score_binder` and `--transform`.
- `flattened_transform` in the CSV is in row-major 4x4 format.
- `target-no-ligand` preprocessing exists (or is generated in this notebook) before delta calculation.

In [3]:
# --------------------- Define key paths -----------------------
import os
import sys
import pandas as pd
import numpy as np

# Path to tricompelx-design repo
tricomplex_design_repo = "/scratch/ymeng/masif-neosurf-af2/tricomplex-design_Meng"

# Paths to MaSIF-Neosurf search results
proj_name = "8VLB_A_3JF_A"
search_af2_grid_dir = os.path.join(tricomplex_design_repo, "search_af2_grid")
search_results_csv = os.path.join(tricomplex_design_repo, "results", "8VLB_A_3JF_A", "8VLB_A_3JF_A_deduplicated.csv")

# Paths to relevant slurm scripts (tricomplex-design repo)
slurm_dir = os.path.join(tricomplex_design_repo, "scripts", "slurm")
slurm_config_path = os.path.join(slurm_dir, "config.sh")

# Path to masif-neosurf repo
masif_neosurf_repo = "/scratch/ymeng/masif-neosurf-af2"

# ---- Load variables defined in config.sh ----
with open(slurm_config_path, "r") as f:
    slurm_config_content = f.read()

# Convert slurm_config_content into python variables
slurm_config = {}
for line in slurm_config_content.split("\n"):
    line = line.strip()
    # Skip comments and blank lines
    if not line or line.startswith('#'):
        continue
    if "=" in line:
        key, value = line.split("=", 1)
        slurm_config[key.strip()] = value.strip()
# ------------------------------------------------

# Preprocessing directories
target_preprocessing_dir = os.path.join(search_af2_grid_dir, proj_name, "preprocessing")
matched_preprocessing_dir = slurm_config["DATA_ROOT"]

___
### Step 1 - End-to-End Scoring Logic (Forward + Reverse + Delta)

This notebook uses `masif_search.py --score_binder --transform` in three stages to estimate ligand dependence.

Required input:
- MaSIF-Neosurf results .csv file
- Preprocessing directory of both `target` and `target_no_ligand`

1. Forward query (sanity check against CSV row)
    - Inputs from one CSV row: `target`, `target_vix`, `matched_protein`, `flattened_transform`.
    - Run score-only mode with:
      - query = `target` at `target_vix`
      - binder = `matched_protein` transformed by `flattened_transform`
    - Purpose: verify we recover row-like scores (`nn_score`, `desc_dist_score`, `mean_desc_dist_score`).

2.  Reverse query (pose-preserving direction swap)
    - Swap roles to score from the matched side back to target:
      - query = `matched_protein` at `matched_vix`
      - binder = `target`
    - Use the inverse transform `invert_transform(flattened_transform)` so the relative pose is preserved after swapping query/binder roles.

3. Delta computation with and without ligand
    - Repeat the same reverse query twice:
      - binder = `target` (with ligand)
      - binder = `target-no-ligand`
    - `compute_delta_scores(...)` automates both runs and returns:
      - prefixed metrics for each condition (`rev_with_ligand_*`, `rev_no_ligand_*`)
      - delta values:
        - `delta_nn_score`
        - `delta_desc_dist_score`
        - `delta_mean_desc_dist_score`


Note: In score-only mode, binder center (`binder_vix`) is chosen by nearest transformed point, while CSV `matched_vix` comes from search-mode descriptor/alignment flow. Therefore `binder_vix` can differ and scores can be close but not identical.

Interpretation guideline: Large positive deltas suggest the matched interaction depends on ligand-associated target features.

In [4]:
# Read search_results_csv and get a key row for testing
df_results = pd.read_csv(search_results_csv)

# Get the row whose uniprot_accn is Q16878 (positive control)
row = df_results[df_results['uniprot_accn'] == 'Q16878'].iloc[0]

pd.set_option('display.max_rows', None)
row

target                                                                     8VLB_A
target_path                     /scratch/ymeng/tricomplex-design_Meng/search_a...
target_site                                                                site_9
target_vix                                                                   2489
matched_protein                                                   Q16878-F1-nD1_A
matched_patch_id                                                               45
matched_protein_path            /scratch/ymeng/tricomplex-design_Meng/search_a...
score                                                                      0.9897
desc_dist_score                                                           30.3785
clashing_ca                                                                     0
clashing_heavy                                                                  0
matched_vix                                                                  2636
desc_dist       

In [5]:
# Reproduce a row score with score-only mode using the saved rigid transform
# (flattened_transform comes from process_search_outputs.py)

import subprocess

# Function to run masif_search.py with score_binder mode
def run_score_binder(
    target_preprocessing_dir,
    target_name,
    target_vix,
    matched_preprocessing_dir,
    matched_protein,
    flattened_transform,
    masif_neosurf_repo,
):
    import subprocess

    cmd = [
        "python",
        f"{masif_neosurf_repo}/masif_search.py",
        # matched_protein args
        "--database", str(matched_preprocessing_dir),
        "--score_binder", str(matched_protein),
        "--transform", str(flattened_transform),
        # target args
        "--target_dir", str(target_preprocessing_dir),
        "--target", str(target_name),
        "--site_vix", str(target_vix),
    ]

    # --- Python 3.6 compatibility: do not use capture_output argument ---
    # Instead, use stdout=subprocess.PIPE, stderr=subprocess.PIPE
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True, check=True)
    print(res.stdout)

    # Parse the final score line into a python dict
    score_line = [line for line in res.stdout.splitlines() if line.startswith("query_name:")][-1]
    score_dict = {}
    for item in score_line.split(", "):
        key, value = item.split(": ", 1)
        try:
            if any(c in value for c in [".", "e", "E"]):
                score_dict[key] = float(value)
            else:
                score_dict[key] = int(value)
        except ValueError:
            score_dict[key] = value

    return score_dict

# Example usage:
# score_dict = run_score_binder(
#     target_preprocessing_dir,
#     target_name,
#     target_vix,
#     matched_preprocessing_dir,
#     matched_protein,
#     flattened_transform,
#     masif_neosurf_repo,
# )
# score_dict


In [6]:
# Try a different row
row = df_results.iloc[0]
row

target                                                                     8VLB_A
target_path                     /scratch/ymeng/tricomplex-design_Meng/search_a...
target_site                                                               site_33
target_vix                                                                   2615
matched_protein                                               A0A024RBG1-F1-nD1_A
matched_patch_id                                                               42
matched_protein_path            /scratch/ymeng/tricomplex-design_Meng/search_a...
score                                                                       0.955
desc_dist_score                                                           25.2485
clashing_ca                                                                     0
clashing_heavy                                                                  5
matched_vix                                                                  2939
desc_dist       

#### Step 1.1 - Reverse Scoring Setup

In this section, we invert `flattened_transform` and swap query/binder roles:
- Query: `matched_protein` at `matched_vix`
- Binder: original `target` (or later `target-no-ligand`)

This keeps the same relative pose while evaluating the target side in reverse mode.

In [7]:
# target and matched_protein details from this row
target_name = row['target']
target_vix = row['target_vix']
matched_protein = row['matched_protein']
matched_protein_vix = row['matched_vix']
flattened_transform = row['flattened_transform']

# Print the expected arguments and values
print(f"query target_preprocessing_dir: {target_preprocessing_dir}")
print(f"query target_name: {target_name}")
print(f"query target_vix: {target_vix}")
print(f"")
print(f"query matched_preprocessing_dir: {matched_preprocessing_dir}")
print(f"query matched_protein: {matched_protein}")
print(f"query flattened_transform: {flattened_transform}")
print(f"")
print(f"Expected CSV score:                {row['score']:.4f}")
print(f"Expected CSV desc_dist_score:      {row['desc_dist_score']:.6f}")
print(f"Expected CSV mean_desc_dist_score: {row['mean_desc_dist_score']:.6f}")

# Example usage:
score_dict = run_score_binder(
    target_preprocessing_dir,
    target_name,
    target_vix,
    matched_preprocessing_dir,
    matched_protein,
    flattened_transform,
    masif_neosurf_repo,
)
score_dict

query target_preprocessing_dir: /scratch/ymeng/masif-neosurf-af2/tricomplex-design_Meng/search_af2_grid/8VLB_A_3JF_A/preprocessing
query target_name: 8VLB_A
query target_vix: 2615

query matched_preprocessing_dir: /work/lpdi/users/diazrovi/domaindome/20260221-AFDBv6_domaindome_DPAM_masif/dpam_domaindome_masif_db
query matched_protein: A0A024RBG1-F1-nD1_A
query flattened_transform: 0.2813929904618225,-0.5241658015415606,0.8037836757568835,89.46002073782068,0.2469589748433733,-0.7698575843218044,-0.5884985680581614,42.21171397049747,0.9272697825677615,0.3641009645091442,-0.08718507888536897,36.330449576716255,0.0,0.0,0.0,1.0

Expected CSV score:                0.9550
Expected CSV desc_dist_score:      25.248450
Expected CSV mean_desc_dist_score: 0.163951
Running surface complementarity search.
Using --transform from inline (det(R)=1.000000, |t|=105.379)
query_name: 8VLB_A, query_site: 0, query_vix: 2615, binder_name: A0A024RBG1-F1-nD1_A, binder_vix: 1566, distance_between_center_points: 

{'query_name': '8VLB_A',
 'query_site': 0,
 'query_vix': 2615,
 'binder_name': 'A0A024RBG1-F1-nD1_A',
 'binder_vix': 1566,
 'distance_between_center_points': 1.352612688665079,
 'query_iface_score': 0.7495731711387634,
 'binder_iface_score': 0.534812867641449,
 'descriptor_distance': 1.958106279373169,
 'nn_score': 0.9080488681793213,
 'desc_dist_score': 24.078268673661917,
 'mean_desc_dist_score': 0.16605702533559943}

In [8]:
# ---- Reverse scoring: query with matched_protein as target to score target ----

# Function to invert flattened_transform
def invert_transform(flattened_transform):
    # Parse the flattened_transform string into a 4x4 matrix
    transform_matrix = np.array(flattened_transform.split(',')).astype(float).reshape(4, 4)
    
    # Invert the 4x4 matrix
    inv_transform_matrix = np.linalg.inv(transform_matrix)
    # Flatten back to a string in the same comma-separated format
    flattened_inv = ','.join(str(x) for x in inv_transform_matrix.flatten())
    return flattened_inv


# target and matched_protein details from this row
target_name = row['target']
target_vix = row['target_vix']
matched_protein = row['matched_protein']
matched_protein_vix = row['matched_vix']
flattened_transform = invert_transform(row['flattened_transform'])

# Print the expected arguments and values
print(f"query target_preprocessing_dir: {matched_preprocessing_dir}")
print(f"query target_name: {matched_protein}")
print(f"query target_vix: {matched_protein_vix}")

print(f"query matched_preprocessing_dir: {target_preprocessing_dir}")
print(f"query matched_protein: {target_name}")
print(f"query flattened_transform: {flattened_transform}")

print(f"Expected CSV score:                {row['score']:.4f}")
print(f"Expected CSV desc_dist_score:      {row['desc_dist_score']:.6f}")
print(f"Expected CSV mean_desc_dist_score: {row['mean_desc_dist_score']:.6f}")


# Run the reverse search
score_dict = run_score_binder(
    target_preprocessing_dir = matched_preprocessing_dir,
    target_name = matched_protein,
    target_vix = matched_protein_vix,
    matched_preprocessing_dir = target_preprocessing_dir,
    matched_protein = target_name,
    flattened_transform = flattened_transform,
    masif_neosurf_repo = masif_neosurf_repo,
)
score_dict

query target_preprocessing_dir: /work/lpdi/users/diazrovi/domaindome/20260221-AFDBv6_domaindome_DPAM_masif/dpam_domaindome_masif_db
query target_name: A0A024RBG1-F1-nD1_A
query target_vix: 2939
query matched_preprocessing_dir: /scratch/ymeng/masif-neosurf-af2/tricomplex-design_Meng/search_af2_grid/8VLB_A_3JF_A/preprocessing
query matched_protein: 8VLB_A
query flattened_transform: 0.28139299046182265,0.2469589748433726,0.9272697825677606,-69.28611245031843,-0.5241658015415604,-0.7698575843218045,0.3641009645091443,66.16093989144132,0.8037836757568826,-0.5884985680581607,-0.08718507888536824,-43.89749796272714,0.0,0.0,0.0,1.0
Expected CSV score:                0.9550
Expected CSV desc_dist_score:      25.248450
Expected CSV mean_desc_dist_score: 0.163951
Running surface complementarity search.
Using --transform from inline (det(R)=1.000000, |t|=105.379)
query_name: A0A024RBG1-F1-nD1_A, query_site: 0, query_vix: 2939, binder_name: 8VLB_A, binder_vix: 1929, distance_between_center_points: 

{'query_name': 'A0A024RBG1-F1-nD1_A',
 'query_site': 0,
 'query_vix': 2939,
 'binder_name': '8VLB_A',
 'binder_vix': 1929,
 'distance_between_center_points': 0.14069621051988776,
 'query_iface_score': 0.5403332114219666,
 'binder_iface_score': 0.7594418525695801,
 'descriptor_distance': 1.9582715034484863,
 'nn_score': 0.9629913568496704,
 'desc_dist_score': 23.46812751818183,
 'mean_desc_dist_score': 0.1641127798474254}

In [ ]:
# Preprocess the target without ligand
target_path = row['target_path']
target_name = f"{row['target']}-no-ligand"

cmd = [
        "python",
        f"{masif_neosurf_repo}/preprocess_pdb.py",
        target_path,
        target_name,
        "-o",target_preprocessing_dir
]

res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True, check=True)
print(res.stdout)

In [10]:
# ---- Score the target without ligand by querying the matched_protein ----

# target and matched_protein details from this row
matched_protein = row['matched_protein']
matched_protein_vix = row['matched_vix']
flattened_transform = row['flattened_transform']

# Print the expected arguments and values
print(f"query target_preprocessing_dir: {matched_preprocessing_dir}")
print(f"query target_name: {matched_protein}")
print(f"query target_vix: {matched_protein_vix}")

print(f"query matched_preprocessing_dir: {target_preprocessing_dir}")
print(f"query matched_protein: {target_name}")
print(f"query flattened_transform: {invert_transform(flattened_transform)}")

print(f"Expected CSV score:                {row['score']:.4f}")
print(f"Expected CSV desc_dist_score:      {row['desc_dist_score']:.6f}")
print(f"Expected CSV mean_desc_dist_score: {row['mean_desc_dist_score']:.6f}")


# Run the reverse search
score_dict = run_score_binder(
    target_preprocessing_dir = matched_preprocessing_dir,
    target_name = matched_protein,
    target_vix = matched_protein_vix,
    matched_preprocessing_dir = target_preprocessing_dir,
    matched_protein = target_name,
    flattened_transform = invert_transform(flattened_transform),
    masif_neosurf_repo = masif_neosurf_repo,
)
score_dict

query target_preprocessing_dir: /work/lpdi/users/diazrovi/domaindome/20260221-AFDBv6_domaindome_DPAM_masif/dpam_domaindome_masif_db
query target_name: A0A024RBG1-F1-nD1_A
query target_vix: 2939
query matched_preprocessing_dir: /scratch/ymeng/masif-neosurf-af2/tricomplex-design_Meng/search_af2_grid/8VLB_A_3JF_A/preprocessing
query matched_protein: 8VLB_A-no-ligand
query flattened_transform: 0.28139299046182265,0.2469589748433726,0.9272697825677606,-69.28611245031843,-0.5241658015415604,-0.7698575843218045,0.3641009645091443,66.16093989144132,0.8037836757568826,-0.5884985680581607,-0.08718507888536824,-43.89749796272714,0.0,0.0,0.0,1.0
Expected CSV score:                0.9550
Expected CSV desc_dist_score:      25.248450
Expected CSV mean_desc_dist_score: 0.163951
Running surface complementarity search.
Using --transform from inline (det(R)=1.000000, |t|=105.379)
query_name: A0A024RBG1-F1-nD1_A, query_site: 0, query_vix: 2939, binder_name: 8VLB_A-no-ligand, binder_vix: 1926, distance_bet

{'query_name': 'A0A024RBG1-F1-nD1_A',
 'query_site': 0,
 'query_vix': 2939,
 'binder_name': '8VLB_A-no-ligand',
 'binder_vix': 1926,
 'distance_between_center_points': 2.4919134684316724,
 'query_iface_score': 0.5403332114219666,
 'binder_iface_score': 0.6007711887359619,
 'descriptor_distance': 2.944021224975586,
 'nn_score': 0.0786719098687172,
 'desc_dist_score': 6.53560404282319,
 'mean_desc_dist_score': 0.08831897355166472}

In [11]:
# Compute reverse-mode delta scores: (with ligand) - (no ligand)
# Query is matched_protein@matched_vix, binder is target or target-no-ligand.

def compute_delta_scores(
    target,
    matched_protein,
    matched_vix,
    flattened_transform,
    target_preprocessing_dir,
    matched_preprocessing_dir,
    masif_neosurf_repo,
):
    required = {
        "target": target,
        "matched_protein": matched_protein,
        "matched_vix": matched_vix,
        "flattened_transform": flattened_transform,
    }
    missing = [k for k, v in required.items() if v is None or (isinstance(v, float) and np.isnan(v))]
    if missing:
        raise ValueError("Missing required arguments: {}".format(", ".join(missing)))

    target_name = str(target)
    matched_protein_name = str(matched_protein)
    try:
        matched_vix_int = int(matched_vix)
    except Exception as exc:
        raise ValueError("matched_vix must be an integer-like value") from exc

    inv_transform = invert_transform(str(flattened_transform))
    target_no_ligand = f"{target_name}-no-ligand"

    try:
        rev_with_ligand = run_score_binder(
            target_preprocessing_dir=matched_preprocessing_dir,
            target_name=matched_protein_name,
            target_vix=matched_vix_int,
            matched_preprocessing_dir=target_preprocessing_dir,
            matched_protein=target_name,
            flattened_transform=inv_transform,
            masif_neosurf_repo=masif_neosurf_repo,
        )
    except Exception as exc:
        raise ValueError("Reverse scoring against ligand target failed") from exc

    try:
        rev_no_ligand = run_score_binder(
            target_preprocessing_dir=matched_preprocessing_dir,
            target_name=matched_protein_name,
            target_vix=matched_vix_int,
            matched_preprocessing_dir=target_preprocessing_dir,
            matched_protein=target_no_ligand,
            flattened_transform=inv_transform,
            masif_neosurf_repo=masif_neosurf_repo,
        )
    except Exception as exc:
        raise ValueError("Reverse scoring against no-ligand target failed") from exc

    result = {
        "target": target_name,
        "target_no_ligand": target_no_ligand,
        "matched_protein": matched_protein_name,
        "matched_vix": matched_vix_int,
    }

    for k, v in rev_with_ligand.items():
        result[f"rev_with_ligand_{k}"] = v
    for k, v in rev_no_ligand.items():
        result[f"rev_no_ligand_{k}"] = v

    result["delta_nn_score"] = float(rev_with_ligand["nn_score"]) - float(rev_no_ligand["nn_score"])
    result["delta_desc_dist_score"] = float(rev_with_ligand["desc_dist_score"]) - float(rev_no_ligand["desc_dist_score"])
    result["delta_mean_desc_dist_score"] = float(rev_with_ligand["mean_desc_dist_score"]) - float(rev_no_ligand["mean_desc_dist_score"])

    return result



In [12]:
# Quick verification on current `row`
target = row["target"]
matched_protein = row["matched_protein"]
matched_vix = row["matched_vix"]
flattened_transform = row["flattened_transform"]

delta_result = compute_delta_scores(
    target=target,
    matched_protein=matched_protein,
    matched_vix=matched_vix,
    flattened_transform=flattened_transform,
    target_preprocessing_dir=target_preprocessing_dir,
    matched_preprocessing_dir=matched_preprocessing_dir,
    masif_neosurf_repo=masif_neosurf_repo,
)

pd.Series(delta_result)


Running surface complementarity search.
Using --transform from inline (det(R)=1.000000, |t|=105.379)
query_name: A0A024RBG1-F1-nD1_A, query_site: 0, query_vix: 2939, binder_name: 8VLB_A, binder_vix: 1929, distance_between_center_points: 0.14069621051988776, query_iface_score: 0.5403332114219666, binder_iface_score: 0.7594418525695801, descriptor_distance: 1.9582715034484863, nn_score: 0.9629913568496704, desc_dist_score: 23.46812751818183, mean_desc_dist_score: 0.1641127798474254

Running surface complementarity search.
Using --transform from inline (det(R)=1.000000, |t|=105.379)
query_name: A0A024RBG1-F1-nD1_A, query_site: 0, query_vix: 2939, binder_name: 8VLB_A-no-ligand, binder_vix: 1926, distance_between_center_points: 2.4919134684316724, query_iface_score: 0.5403332114219666, binder_iface_score: 0.6007711887359619, descriptor_distance: 2.944021224975586, nn_score: 0.0786719098687172, desc_dist_score: 6.53560404282319, mean_desc_dist_score: 0.08831897355166472



target                                                         8VLB_A
target_no_ligand                                     8VLB_A-no-ligand
matched_protein                                   A0A024RBG1-F1-nD1_A
matched_vix                                                      2939
rev_with_ligand_query_name                        A0A024RBG1-F1-nD1_A
rev_with_ligand_query_site                                          0
rev_with_ligand_query_vix                                        2939
rev_with_ligand_binder_name                                    8VLB_A
rev_with_ligand_binder_vix                                       1929
rev_with_ligand_distance_between_center_points               0.140696
rev_with_ligand_query_iface_score                            0.540333
rev_with_ligand_binder_iface_score                           0.759442
rev_with_ligand_descriptor_distance                           1.95827
rev_with_ligand_nn_score                                     0.962991
rev_with_ligand_desc

___
### Step 2 - Run the Same Workflow via CLI (`ligand_delta_scores.py`)

The script `ligand_delta_scores.py` packages the reverse-scoring delta workflow from Step 1 into a single command.

What it does:
1. Builds `target_no_ligand = f"{target_name}-no-ligand"`.
2. Checks whether no-ligand descriptors already exist under `target_preproc_dir`.
3. If missing, runs `preprocess_pdb.py` (without ligand args) to preprocess `target_no_ligand`.
4. Runs reverse score-only mode twice using `masif_search.py --score_binder --transform`:
   - binder = `target_name`
   - binder = `target_no_ligand`
5. Computes and reports:
   - `delta_nn_score`
   - `delta_desc_dist_score`
   - `delta_mean_desc_dist_score`

Useful flags:
- `--output`: write a one-row CSV in addition to stdout JSON.
- `--verbose`: print subprocess commands and their stdout/stderr for debugging.



In [13]:
# Example: run ligand_delta_scores.py on the current row

target_name = row["target"]
match_name = row["matched_protein"]
match_vix = int(row["matched_vix"])
flattened_transform = row["flattened_transform"]

!python {masif_neosurf_repo}/ligand_delta_scores.py \
    --target_preproc_dir "{target_preprocessing_dir}" \
    --target_name "{target_name}" \
    --match_preproc_dir "{matched_preprocessing_dir}" \
    --match_name "{match_name}" \
    --match_vix "{match_vix}" \
    --flattened_transform "{flattened_transform}" \
    --masif-neosurf-repo "{masif_neosurf_repo}" \
    --verbose


[run_score_binder] Command:
/usr/local/bin/python /scratch/ymeng/masif-neosurf-af2/masif_search.py --database /scratch/ymeng/masif-neosurf-af2/tricomplex-design_Meng/search_af2_grid/8VLB_A_3JF_A/preprocessing --score_binder 8VLB_A --transform 0.28139299046182265,0.2469589748433726,0.9272697825677606,-69.28611245031843,-0.5241658015415604,-0.7698575843218045,0.3641009645091443,66.16093989144132,0.8037836757568826,-0.5884985680581607,-0.08718507888536824,-43.89749796272714,0.0,0.0,0.0,1.0 --target_dir /work/lpdi/users/diazrovi/domaindome/20260221-AFDBv6_domaindome_DPAM_masif/dpam_domaindome_masif_db --target A0A024RBG1-F1-nD1_A --site_vix 2939
[run_score_binder] STDOUT:
Running surface complementarity search.
Using --transform from inline (det(R)=1.000000, |t|=105.379)
query_name: A0A024RBG1-F1-nD1_A, query_site: 0, query_vix: 2939, binder_name: 8VLB_A, binder_vix: 1929, distance_between_center_points: 0.14069621051988776, query_iface_score: 0.5403332114219666, binder_iface_score: 0.7594